In [2]:
import pandas as pd
import drawbridge
from drawbridge.models.strain import Strain
from drawbridge.models.avro.dna_component import DnaComponent
from drawbridge.codon.locate import annotations_typed
from drawbridge.codon.locate import annotations_named

In [2]:
drawbridge.connect()

In [3]:
df = pd.read_csv("~/Downloads/FO_dummy_data.csv")

In [8]:
df.iloc[0].dna_component_id 

13004907109.0

In [4]:
def df_to_fasta(
    df, 
    strain_id_col="strain_id", 
    dna_component_id_col="dna_component_id",
    genome_component_name="replicating_plasmid",
    output_filename = "sequences.fasta"
):    

    # Ensure requried columns are present.
    if dna_component_id_col not in df:
        df[dna_component_id_col] = None
    for required_col in [strain_id_col, dna_component_id_col]:
        if required_col not in df:
            raise ValueError(f"Columns {required_col} is required.")
    
    # Remove redundent rows.
    df = df[[strain_id_col, dna_component_id_col]].drop_duplicates()
    
    print(f"Converting {len(df)} strains to fasta")    
    print(f"Using strain {int(df.iloc[0][strain_id_col])} as the reference.")
    
    # Iterate over rows
    fasta_data = ""
    for i, row in df.iterrows():
        try:
            strain_id = int(row[strain_id_col])
            dna_component_id = row[dna_component_id_col] 
            
            if dna_component_id > 0:
                # Use manually provided dna component if present. 
                dna = DnaComponent.find(int(dna_component_id))
            else:
                # Otherwise use the dna component associated with the strain
                s = Strain.find(strain_id)
                if s.genomeComponents and genome_component_name in s.genomeComponents:
                    dna = DnaComponent.find(s.genomeComponents[genome_component_name])
                else:
                    raise Exception(f"A {genome_component_name} is not associated with this strian's genome.")
            
            # Extract the cds region by annotation
            # HACK: Use the largest cds region if there are multiple.
            cds = sorted(annotations_typed('CDS')(dna), key=len)[-1].found_component
            
            fasta_data += f">{strain_id}\n"
            fasta_data += f"{cds.sequence}\n"
            
        except Exception as e:
            print(f"Warning: Failed to process row {i}: strain_id: {strain_id}. {e}")
    
    # TODO(flash): It might be slightly better to write data incrementally, but whatevs.
    if output_filename:
        print(f"Writing results to {output_filename}")
        with open(output_filename, 'w') as f:
            f.write(fasta_data)
    
    return fasta_data

In [5]:
out =  df_to_fasta(df)

Converting 471 strains to fasta
Using strain 7000856232 as the reference.
Writing results to sequences.fasta


In [9]:
None > 0

TypeError: '>' not supported between instances of 'NoneType' and 'int'